# AES, from scratch

[NIST FIPS 197](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.197.pdf)


[This lecture notes are good](https://www.cs.uaf.edu/2015/spring/cs463/lecture/03_23_AES.html)

## Terminology and Notation

`Key Expansion` -- Routine used to generate a series of Round Keys from the Cipher Key. 

`Cipher Key` -- Secret, cryptographic key that is used by the Key Expansion routine to generate a set of Round Keys; can be pictured as a rectangular array of bytes, having four rows and $Nk$ columns.

`Round Key` -- Round keys are values derived from the Cipher Key using the Key Expansion routine; they are applied to the State in the Cipher and Inverse Cipher. 

`State` -- Intermediate Cipher result that can be pictured as a rectangular array of bytes, having four rows and $Nb$ columns. 


$Nb$ -- Number of columns (32-bit words) comprising the `State`. For AES $Nb = 4$.

$Nk$ -- Number of 32-bit words comprising the `Cipher Key`. For this standard $Nk = 4$, $Nk = 6$ or $Nk = 8$.

$Nr$ -- Number of rounds, which is a function of $Nk$ and $Nb$. For this standard $Nr = 10$, $Nr = 12$ or $Nr = 14$.

In [1]:
OVERFLOW_MASK = 0x100
IRR_POLY = 0x11B


class ff(object):
    def __init__(self, element):
        assert element <= 0xFF
        self.e = element

    def __add__(self, other):
        return ff(self.e ^ other.e)

    def __mul__(self, other):
        l, r = self.e, other.e
        p = 0
        while r:
            if r & 1:
                p ^= l
            l <<= 1
            if l & OVERFLOW_MASK:
                l ^= IRR_POLY
            r >>= 1
        return ff(p & OVERFLOW_MASK - 1)

    def __eq__(self, other):
        return self.e == other.e


# FIPS 197 Section 4.1
assert ff(0x57) + ff(0x83) == ff(0xD4)

# FIPS 197 Section 4.2
assert ff(0x57) * ff(0x83) == ff(0xC1)

# "The multiplication defined above is associative,
# and the element {01} is the multiplicative
# identity."

assert ff(0x66) * ff(0x01) == ff(0x66)
assert ff(0x55) * ff(0x66) == ff(0x66) * ff(0x55)

In [2]:
sbox = (
    0x63, 0x7C, 0x77, 0x7B, 0xF2, 0x6B, 0x6F, 0xC5, 0x30, 0x01, 0x67, 0x2B, 0xFE, 0xD7, 0xAB, 0x76,
    0xCA, 0x82, 0xC9, 0x7D, 0xFA, 0x59, 0x47, 0xF0, 0xAD, 0xD4, 0xA2, 0xAF, 0x9C, 0xA4, 0x72, 0xC0,
    0xB7, 0xFD, 0x93, 0x26, 0x36, 0x3F, 0xF7, 0xCC, 0x34, 0xA5, 0xE5, 0xF1, 0x71, 0xD8, 0x31, 0x15,
    0x04, 0xC7, 0x23, 0xC3, 0x18, 0x96, 0x05, 0x9A, 0x07, 0x12, 0x80, 0xE2, 0xEB, 0x27, 0xB2, 0x75,
    0x09, 0x83, 0x2C, 0x1A, 0x1B, 0x6E, 0x5A, 0xA0, 0x52, 0x3B, 0xD6, 0xB3, 0x29, 0xE3, 0x2F, 0x84,
    0x53, 0xD1, 0x00, 0xED, 0x20, 0xFC, 0xB1, 0x5B, 0x6A, 0xCB, 0xBE, 0x39, 0x4A, 0x4C, 0x58, 0xCF,
    0xD0, 0xEF, 0xAA, 0xFB, 0x43, 0x4D, 0x33, 0x85, 0x45, 0xF9, 0x02, 0x7F, 0x50, 0x3C, 0x9F, 0xA8,
    0x51, 0xA3, 0x40, 0x8F, 0x92, 0x9D, 0x38, 0xF5, 0xBC, 0xB6, 0xDA, 0x21, 0x10, 0xFF, 0xF3, 0xD2,
    0xCD, 0x0C, 0x13, 0xEC, 0x5F, 0x97, 0x44, 0x17, 0xC4, 0xA7, 0x7E, 0x3D, 0x64, 0x5D, 0x19, 0x73,
    0x60, 0x81, 0x4F, 0xDC, 0x22, 0x2A, 0x90, 0x88, 0x46, 0xEE, 0xB8, 0x14, 0xDE, 0x5E, 0x0B, 0xDB,
    0xE0, 0x32, 0x3A, 0x0A, 0x49, 0x06, 0x24, 0x5C, 0xC2, 0xD3, 0xAC, 0x62, 0x91, 0x95, 0xE4, 0x79,
    0xE7, 0xC8, 0x37, 0x6D, 0x8D, 0xD5, 0x4E, 0xA9, 0x6C, 0x56, 0xF4, 0xEA, 0x65, 0x7A, 0xAE, 0x08,
    0xBA, 0x78, 0x25, 0x2E, 0x1C, 0xA6, 0xB4, 0xC6, 0xE8, 0xDD, 0x74, 0x1F, 0x4B, 0xBD, 0x8B, 0x8A,
    0x70, 0x3E, 0xB5, 0x66, 0x48, 0x03, 0xF6, 0x0E, 0x61, 0x35, 0x57, 0xB9, 0x86, 0xC1, 0x1D, 0x9E,
    0xE1, 0xF8, 0x98, 0x11, 0x69, 0xD9, 0x8E, 0x94, 0x9B, 0x1E, 0x87, 0xE9, 0xCE, 0x55, 0x28, 0xDF,
    0x8C, 0xA1, 0x89, 0x0D, 0xBF, 0xE6, 0x42, 0x68, 0x41, 0x99, 0x2D, 0x0F, 0xB0, 0x54, 0xBB, 0x16
)

# 5.1.1
assert sbox[0x53] == 0xED

In [3]:
# 5.1.2


def rotate(arr, n):
    assert len(arr) == 4
    n %= len(arr)
    return arr[n:] + arr[:n]


def shift_rows(state):
    assert len(state) == 4
    return [
        rotate(state[0], 0),
        rotate(state[1], 1),
        rotate(state[2], 2),
        rotate(state[3], 3),
    ]


assert rotate([1, 2, 3, 4], 3) == [4, 1, 2, 3]
assert rotate([1, 2, 3, 4], 4) == [1, 2, 3, 4]

after = shift_rows(
    [
        ["s00", "s01", "s02", "s03"],
        ["s10", "s11", "s12", "s13"],
        ["s20", "s21", "s22", "s23"],
        ["s30", "s31", "s32", "s33"],
    ]
)

assert after == [
    ["s00", "s01", "s02", "s03"],
    ["s11", "s12", "s13", "s10"],
    ["s22", "s23", "s20", "s21"],
    ["s33", "s30", "s31", "s32"],
]

In [4]:
# 5.1.3


def _mix_column(s0c, s1c, s2c, s3c):
    two, three = ff(0x02), ff(0x03)
    return (
        (two * s0c) + (three * s1c) + s2c + s3c,  # s'0c
        s0c + (two * s1c) + (three * s2c) + s3c,  # s'1c
        s0c + s1c + (two * s2c) + (three * s3c),  # s'2c
        (three * s0c) + s1c + s2c + (two * s3c),  # s'3c
    )


# From test vectors here: https://en.wikipedia.org/wiki/Rijndael_MixColumns
assert _mix_column(ff(0xDB), ff(0x13), ff(0x53), ff(0x45)) == (
    ff(0x8E),
    ff(0x4D),
    ff(0xA1),
    ff(0xBC),
)


def mix_columns(state):
    # column 0
    sp00, sp10, sp20, sp30 = _mix_column(
        state[0][0], state[1][0], state[2][0], state[3][0]
    )
    # column 1
    sp01, sp11, sp21, sp31 = _mix_column(
        state[0][1], state[1][1], state[2][1], state[3][1]
    )
    # column 2
    sp02, sp12, sp22, sp32 = _mix_column(
        state[0][2], state[1][2], state[2][2], state[3][2]
    )
    # column 3
    sp03, sp13, sp23, sp33 = _mix_column(
        state[0][3], state[1][3], state[2][3], state[3][3]
    )
    return [
        [sp00, sp01, sp02, sp03],
        [sp10, sp11, sp12, sp13],
        [sp20, sp21, sp22, sp23],
        [sp30, sp31, sp32, sp33],
    ]

In [5]:
# 5.1.4


def add_round_key(state, rkey):
    return [
        [
            state[0][0] + rkey[0][0],
            state[0][1] + rkey[0][1],
            state[0][2] + rkey[0][2],
            state[0][3] + rkey[0][3],
        ],
        [
            state[1][0] + rkey[1][0],
            state[1][1] + rkey[1][1],
            state[1][2] + rkey[1][2],
            state[1][3] + rkey[1][3],
        ],
        [
            state[2][0] + rkey[2][0],
            state[2][1] + rkey[2][1],
            state[2][2] + rkey[2][2],
            state[2][3] + rkey[2][3],
        ],
        [
            state[3][0] + rkey[3][0],
            state[3][1] + rkey[3][1],
            state[3][2] + rkey[3][2],
            state[3][3] + rkey[3][3],
        ],
    ]

In [6]:
rcon = [
    list(map(ff, [0x01, 0x00, 0x00, 0x00])),
    list(map(ff, [0x02, 0x00, 0x00, 0x00])),
    list(map(ff, [0x04, 0x00, 0x00, 0x00])),
    list(map(ff, [0x08, 0x00, 0x00, 0x00])),
    list(map(ff, [0x10, 0x00, 0x00, 0x00])),
    list(map(ff, [0x20, 0x00, 0x00, 0x00])),
    list(map(ff, [0x40, 0x00, 0x00, 0x00])),
    list(map(ff, [0x80, 0x00, 0x00, 0x00])),
    list(map(ff, [0x1B, 0x00, 0x00, 0x00])),
    list(map(ff, [0x36, 0x00, 0x00, 0x00])),
]


def xor_word(lword, rword):
    assert len(lword) == 4 and len(lword) == len(rword)
    return [
        lword[0] + rword[0],
        lword[1] + rword[1],
        lword[2] + rword[2],
        lword[3] + rword[3],
    ]


def rot_word(word):
    assert len(word) == 4
    return rotate(word, 1)

def rot_words(words):
    assert len(words) == 4
    return [
        rot_word(words[0]),
        rot_word(words[1]),
        rot_word(words[2]),
        rot_word(words[3]),
    ]


def sub_word(word):
    assert len(word) == 4
    return [
        ff(sbox[word[0].e]),
        ff(sbox[word[1].e]),
        ff(sbox[word[2].e]),
        ff(sbox[word[3].e]),
    ]

def sub_words(words):
    assert len(words) == 4
    return [
        sub_word(words[0]),
        sub_word(words[1]),
        sub_word(words[2]),
        sub_word(words[3]),
    ]

def key_schedule(k):
    assert len(k) == 16
    n = 4  # Word length
    r = 11  # Round length

    words = [
        [ff(k[0]), ff(k[1]), ff(k[2]), ff(k[3])],
        [ff(k[4]), ff(k[5]), ff(k[6]), ff(k[7])],
        [ff(k[8]), ff(k[9]), ff(k[10]), ff(k[11])],
        [ff(k[12]), ff(k[13]), ff(k[14]), ff(k[15])],
    ]

    for i in range(n, 4 * r):
        words.append([])
        if i % n == 0:
            words[i] = xor_word(words[i - n], sub_word(rot_word(words[i - 1])))
            words[i] = xor_word(words[i], rcon[i // n - 1])
        elif n > 6 and i % n == 4:
            words[i] = xor_word(words[i - n], sub_word(words[i - 1]))
        else:
            words[i] = xor_word(words[i - n], words[i - 1])
    return words


def pretty_expanded(k):
    from binascii import hexlify

    exp = key_schedule(k)
    for i in range(11):
        print(
            hexlify(
                bytes(exp[4 * i] + exp[4 * i + 1] + exp[4 * i + 2] + exp[4 * i + 3])
            )
        )


assert rot_word(["b0", "b1", "b2", "b3"]) == ["b1", "b2", "b3", "b0"]

In [9]:
from binascii import hexlify

exp = key_schedule(b"\x00" * 16)
assert (
    hexlify(bytes(map(lambda ff: ff.e, exp[40] + exp[41] + exp[42] + exp[43])))
    == b"b4ef5bcb3e92e21123e951cf6f8f188e"
)

In [12]:
def aes128(key, block):
    assert len(key) == 16 and len(key) == len(block)
    expanded = key_schedule(key)

    # round 0
    ffblock = list(map(ff, block))
    state = add_round_key(
        expanded[0:4],
        [
            [ffblock[0], ffblock[1], ffblock[2], ffblock[3]],
            [ffblock[4], ffblock[5], ffblock[6], ffblock[7]],
            [ffblock[8], ffblock[9], ffblock[10], ffblock[11]],
            [ffblock[12], ffblock[13], ffblock[14], ffblock[15]],
        ],
    )

    # round 1 -> round 9
    for i in range(1, 10):
        state = sub_words(state)
        state = shift_rows(state)
        state = mix_columns(state)
        state = add_round_key(state, expanded[i])

    # round 10
    state = sub_word(state)
    state = shift_rows(state)
    state = add_round_key(state, expanded[10])
    return bytes(state[0] + state[1] + state[2] + state[3])

# Test Vectors

https://csrc.nist.gov/CSRC/media/Projects/Cryptographic-Standards-and-Guidelines/documents/examples/AES_Core128.pdf

In [13]:
from binascii import unhexlify

key = unhexlify("2B7E151628AED2A6ABF7158809CF4F3C")
block = unhexlify("6BC1BEE22E409F96E93D7E117393172A")
ciphertext = unhexlify("3AD77BB40D7A3660A89ECAF32466EF97")
assert len(key) == 16 and len(block) == 16
assert aes128(key, block) == ciphertext

TypeError: 'ff' object is not subscriptable